<a href="https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 414, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 414 (delta 183), reused 110 (delta 100), pack-reused 178 (from 2)
Receiving objects: 100% (414/414), 1.97 MiB | 7.17 MiB/s, done.
Resolving deltas: 100% (238/238), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [21]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Distributions

Key fields checked for shape before trusting them in any rule: `ctr` and
`impressions_90d` both show heavy right tails — a small number of very
high-traffic pages dominate the mean, so median and percentiles matter more
than the raw mean for these.

## 1. Distributions

Key fields checked for shape before trusting them in any rule: `ctr` and
`impressions_90d` both show heavy right tails — a small number of very
high-traffic pages dominate the mean, so median and percentiles matter more
than the raw mean for these.

In [22]:
print(df[['ctr', 'impressions_90d', 'avg_position', 'days_since_last_update']].describe())

zero_ctr = (df['ctr'] == 0).sum()
print(f"\nrows with ctr == 0: {zero_ctr} ({zero_ctr/len(df):.1%} of dataset)")

                ctr  impressions_90d  avg_position  days_since_last_update
count  30000.000000     30000.000000   30000.00000            30000.000000
mean       0.510733      5200.366300      16.34238               46.098300
std        3.279162     16838.019547      15.21679               42.078709
min        0.000000         1.000000       0.00000                1.000000
25%        0.000000        81.000000       6.20000               20.000000
50%        0.070000       731.000000      10.80000               20.000000
75%        0.290000      3615.250000      22.30000              104.000000
max      100.000000    517715.000000     245.00000              373.000000

rows with ctr == 0: 13212 (44.0% of dataset)


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## 2. Signal test #1 / #2 / #3

Three signals tested against the observed decline rate, each with a verdict.

In [23]:

sig1 = df.groupby('freshness_tier')['is_declining'].agg(['mean', 'count'])
sig1.columns = ['decline_rate', 'n']
print("Signal 1 — freshness_tier:")
print(sig1)
print("Verdict: MIXED — decline rises with staleness through 91-180d, then reverses for 181+.\n")

# Signal 2: CTR vs. position-tier peers
median_ctr_by_tier = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = median_ctr_by_tier - df['ctr']
sig2 = df.groupby(df['ctr_gap'] > 0)['is_declining'].agg(['mean', 'count'])
sig2.index = ['above_or_equal_median', 'below_median']
sig2.columns = ['decline_rate', 'n']
print("Signal 2 — CTR vs. position peers:")
print(sig2)
print("Verdict: CONFIRMED — clear, consistent gap on large samples.\n")

# Signal 3: content age
sig3 = df.groupby(
    pd.cut(df['content_age_days'], bins=[0,90,180,365,10000]),
    observed=False   # explicitly set to silence FutureWarning
)['is_declining'].agg(['mean','count'])
sig3.columns = ['decline_rate', 'n']
print("Signal 3 — content_age_days:")
print(sig3)


Signal 1 — freshness_tier:
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
181+                0.471264    174
31-90               0.588571    175
91-180              0.611057   9171
Verdict: MIXED — decline rises with staleness through 91-180d, then reverses for 181+.

Signal 2 — CTR vs. position peers:
                       decline_rate      n
above_or_equal_median      0.502630  16921
below_median               0.593088  13079
Verdict: CONFIRMED — clear, consistent gap on large samples.

Signal 3 — content_age_days:
                  decline_rate      n
content_age_days                     
(0, 90]               0.668699    492
(90, 180]             0.625552  11780
(180, 365]            0.514866  11368
(365, 10000]          0.426258   6360


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. The flag-linked test

Staleness (`freshness_tier`) is the signal behind FlyRank's real refresh flags —
already tested above in Signal 1. The rule's assumption ("older = more likely
declining") holds only partially: true through 91-180 days, but reverses for
181+, meaning very old untouched content behaves more like stable evergreen
content than neglected content.

In [24]:
print(sig1)

                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
181+                0.471264    174
31-90               0.588571    175
91-180              0.611057   9171


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## 4. What this means in practice

A content team using staleness alone as a refresh trigger should cap it at
~180 days, not treat "older = more urgent" as a straight line — content that's
survived 6+ months untouched is more often stable than at-risk. CTR-vs-position
is the more reliable standalone signal of the three tested here, and pairing it
with capped staleness (not raw staleness) gives a more honest rule than either
alone.

In [25]:
print("No query needed — this section is a practical takeaway, not a data check.")

No query needed — this section is a practical takeaway, not a data check.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.